#  FinNavigator: Qwen3-VL Fine-Tuning
This notebook provides a clean, consolidated pipeline for fine-tuning the **Qwen3-VL-4B** model using **Unsloth**. It integrates baseline SEC data, fundamental stock metrics, and synthetic career strategy data, tracks experiments with **TrackIO**, and exports GGUF for local execution.

## 1. Environment & API Setup

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets xformers
!pip install -q trackio wandb alpha_vantage sec-api requests pandas

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
import trackio
import wandb

# 1. HuggingFace Login
hf_token = userdata.get('HF_TOKEN')
if hf_token: login(token=hf_token)

# 2. WandB Login
wandb_api_key = userdata.get('WANDB_API_KEY')
if wandb_api_key: wandb.login(key=wandb_api_key)

# 3. TrackIO Initialization
run = trackio.init(project="FinNavigator-LLM", run_name="Qwen-3-VL-4B-Instruct-SFT")

# 4. SEC & Finance Keys
AV_KEY = userdata.get('ALPHA_VANTAGE_KEY')
STOCKFIT_KEY = userdata.get('StockFit_API_KEY') # Used for SEC-API

## 📊 2. Data Preparation
We collect live data and merge it with local synthetic career data.

### 2.1 Live Data Collection (SEC & Alpha Vantage)
Use these functions to fetch additional training samples directly in Colab.

In [ ]:
import requests
import time
from sec_api import QueryApi, ExtractorApi

def fetch_sec_section(ticker, section="1A"):
    """Pull a single section of the latest 10-K via sec-api."""
    if not STOCKFIT_KEY: 
        print("⚠️ Missing StockFit_API_KEY (SEC-API)")
        return None
    try:
        q = QueryApi(api_key=STOCKFIT_KEY)
        e = ExtractorApi(api_key=STOCKFIT_KEY)
        resp = q.get_filings({
            "query": f'ticker:{ticker} AND formType:"10-K"',
            "from": "0", "size": "1",
            "sort": [{"filedAt": {"order": "desc"}}],
        })
        filings = resp.get("filings", [])
        if not filings: return None
        url = filings[0].get("linkToFilingDetails")
        return e.get_section(url, section, "text")
    except Exception as exc:
        print(f"[SEC Error] {ticker}: {exc}")
        return None

def fetch_av_overview(ticker):
    """Pull company overview from Alpha Vantage."""
    if not AV_KEY: 
        print("⚠️ Missing ALPHA_VANTAGE_KEY")
        return None
    try:
        r = requests.get(
            "https://www.alphavantage.co/query",
            params={"function": "OVERVIEW", "symbol": ticker, "apikey": AV_KEY},
            timeout=15
        )
        data = r.json()
        time.sleep(13) # AV free tier limit: 5 calls/min
        return data if "Symbol" in data else None
    except Exception as exc:
        print(f"[AV Error] {ticker}: {exc}")
        return None

### 2.2 Formatting & Merging Logic

In [ ]:
import json
from datasets import load_dataset

def format_prompts(examples, tokenizer_eos):
    texts = []
    ALPACA_PROMPT = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""
    for instruction, input_text, output in zip(examples["instruction"], examples["input"], examples["output"]):
        text = ALPACA_PROMPT.format(instruction=instruction, input=input_text, output=output)
        texts.append(text + tokenizer_eos)
    return {"text": texts}

def merge_datasets():
    # List of files to check in /content/
    data_files = ["finnav_train.jsonl", "synthetic_career_data.jsonl"]
    existing_files = [f for f in data_files if os.path.exists(f)]
    
    if not existing_files:
        print("⚠️ No data files found. Please upload finnav_train.jsonl or synthetic_career_data.jsonl")
        return None
        
    dataset = load_dataset("json", data_files=existing_files, split="train")
    print(f"✅ Loaded {len(dataset)} examples from {existing_files}")
    return dataset

## 🤖 3. Model Configuration (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
DTYPE = None
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-VL-4B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 🔥 4. Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

class TrackIOCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs: run.log(logs, step=state.global_step)

raw_dataset = merge_datasets()
if raw_dataset:
    formatted_ds = raw_dataset.map(lambda x: format_prompts(x, tokenizer.eos_token), batched=True)
    ds_split = formatted_ds.train_test_split(test_size=0.1, seed=42)

    args = TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        evaluation_strategy="steps",
        eval_steps=10,
        optim="adamw_8bit",
        report_to="wandb",
        seed=3407,
    )

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, 
        train_dataset=ds_split["train"], eval_dataset=ds_split["test"],
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        args=args, callbacks=[TrackIOCallback()]
    )

    trainer.train()
    run.finish()

## 💾 5. Save & Export (GGUF)

In [ ]:
SAVE_NAME = "finnav_qwen3.5_4b_lora"
model.save_pretrained(SAVE_NAME)
tokenizer.save_pretrained(SAVE_NAME)

# Push to Hub
model.push_to_hub(f"MOH749/{SAVE_NAME}")
tokenizer.push_to_hub(f"MOH749/{SAVE_NAME}")

# Export to GGUF
model.save_pretrained_gguf(
    f"{SAVE_NAME}_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)